In [17]:
import pandas as pd
import itertools

# 1. Funcoes auxiliares de jogo
def verifica_vencedor(board):
    linhas_vitoria = [
        [0, 1, 2], [3, 4, 5], [6, 7, 8],  # Horizontais
        [0, 3, 6], [1, 4, 7], [2, 5, 8],  # Verticais
        [0, 4, 8], [2, 4, 6]              # Diagonais
    ]
    vence_x = False
    vence_o = False

    for l in linhas_vitoria:
        if board[l[0]] == board[l[1]] == board[l[2]] == 1:
            vence_x = True
        elif board[l[0]] == board[l[1]] == board[l[2]] == -1:
            vence_o = True

    return vence_x, vence_o


def existe_fim_na_proxima_jogada(state):
    count_x = state.count(1)
    count_o = state.count(-1)

    # Quem joga agora: X quando as contagens sao iguais, senao O.
    jogador = 1 if count_x == count_o else -1

    for i, v in enumerate(state):
        if v != 0:
            continue

        proximo_estado = list(state)
        proximo_estado[i] = jogador

        vence_x, vence_o = verifica_vencedor(proximo_estado)
        if vence_x or vence_o or 0 not in proximo_estado:
            return True

    return False


# 2. Gerar TODAS as 19.683 combinacoes possiveis (1 = X, -1 = O, 0 = vazio)
opcoes = [1, -1, 0]
todas_combinacoes = list(itertools.product(opcoes, repeat=9))

estados_validos = []

# 3. Filtrar usando as regras matematicas do Jogo da Velha
for state in todas_combinacoes:
    count_x = state.count(1)
    count_o = state.count(-1)
    count_vazio = state.count(0)

    # Regra A: Como X comeca, a quantidade de X tem que ser igual a O
    # ou exatamente 1 a mais que O.
    if count_x != count_o and count_x != count_o + 1:
        continue

    vence_x, vence_o = verifica_vencedor(state)

    # Regra B: X e O nao podem vencer ao mesmo tempo em jogo valido.
    if vence_x and vence_o:
        continue

    # Regra C: Se X venceu, ele deve ter 1 peca a mais.
    if vence_x and count_x != count_o + 1:
        continue

    # Regra D: Se O venceu, as quantidades devem ser iguais.
    if vence_o and count_x != count_o:
        continue

    # 4. Classificacao exata do estado
    if vence_x:
        label = 'Fim_X_Vence'
    elif vence_o:
        label = 'Fim_O_Vence'
    elif count_vazio == 0:
        label = 'Fim_Empate'
    else:
        total_pecas = count_x + count_o

        # Nova categoria pedida: pelo menos 1 peca e sem fim possivel na proxima jogada.
        if total_pecas >= 1 and not existe_fim_na_proxima_jogada(state):
            label = 'Tem jogo'
        else:
            label = 'Possibilidade_fim'

    estados_validos.append(list(state) + [label])

# Criar o DataFrame e conferir o resultado
df_completo = pd.DataFrame(estados_validos)

# Salvar e exibir as contagens
df_completo.to_csv('dataset_tratado_balanceado.csv', index=False, header=False)

print('Geracao concluida de forma combinatoria!')
print('\nDistribuicao dos cenarios possiveis:')
print(df_completo[9].value_counts())

Geracao concluida de forma combinatoria!

Distribuicao dos cenarios possiveis:
9
Possibilidade_fim    2439
Tem jogo             2081
Fim_X_Vence           626
Fim_O_Vence           316
Fim_Empate             16
Name: count, dtype: int64


In [18]:
import pandas as pd
import itertools

# Setor dedicado: popular o dataset somente com casos 'Tem jogo' sem simulacao de partidas.
linhas_vitoria = [
    [0, 1, 2], [3, 4, 5], [6, 7, 8],
    [0, 3, 6], [1, 4, 7], [2, 5, 8],
    [0, 4, 8], [2, 4, 6]
]


def possui_vitoria(board):
    for l in linhas_vitoria:
        soma = board[l[0]] + board[l[1]] + board[l[2]]
        if soma == 3 or soma == -3:
            return True
    return False


def estado_e_valido(board):
    count_x = board.count(1)
    count_o = board.count(-1)

    if count_x != count_o and count_x != count_o + 1:
        return False

    vence_x = False
    vence_o = False
    for l in linhas_vitoria:
        soma = board[l[0]] + board[l[1]] + board[l[2]]
        if soma == 3:
            vence_x = True
        elif soma == -3:
            vence_o = True

    if vence_x and vence_o:
        return False
    if vence_x and count_x != count_o + 1:
        return False
    if vence_o and count_x != count_o:
        return False

    return True


def eh_tem_jogo_na_marra(board):
    count_x = board.count(1)
    count_o = board.count(-1)
    vazios = board.count(0)

    # Pelo menos uma peca no tabuleiro.
    if count_x + count_o < 1:
        return False

    # Se tiver 1 vazio, a proxima jogada necessariamente encerra em vitoria ou empate.
    if vazios < 2:
        return False

    if possui_vitoria(board):
        return False

    jogador_da_vez = 1 if count_x == count_o else -1

    # Sem simulacao: basta checar se existe linha com 2 pecas do jogador da vez + 1 vazio.
    # Se existir, a proxima jogada pode terminar em vitoria.
    for l in linhas_vitoria:
        valores = [board[i] for i in l]
        if valores.count(jogador_da_vez) == 2 and valores.count(0) == 1:
            return False

    return True


arquivo_dataset = 'dataset_tratado_balanceado.csv'
df_existente = pd.read_csv(arquivo_dataset, header=None)

# Conjunto de tabuleiros ja presentes para evitar repeticao.
tabuleiros_existentes = {
    tuple(int(v) for v in linha[:9])
    for linha in df_existente.itertuples(index=False, name=None)
}

novos_tem_jogo = []
for board in itertools.product([1, -1, 0], repeat=9):
    board_list = list(board)

    if not estado_e_valido(board_list):
        continue

    if not eh_tem_jogo_na_marra(board_list):
        continue

    board_tuple = tuple(board_list)
    if board_tuple in tabuleiros_existentes:
        continue

    novos_tem_jogo.append(board_list + ['Tem jogo'])
    tabuleiros_existentes.add(board_tuple)


df_novos = pd.DataFrame(novos_tem_jogo)

if not df_novos.empty:
    df_resultado = pd.concat([df_existente, df_novos], ignore_index=True)
    # Garantia final de unicidade por configuracao de tabuleiro (primeiras 9 colunas).
    df_resultado = df_resultado.drop_duplicates(subset=list(range(9)), keep='first')
    df_resultado.to_csv(arquivo_dataset, index=False, header=False)

print(f'Novos casos Tem jogo adicionados: {len(df_novos)}')
print(f'Total final de linhas: {len(pd.read_csv(arquivo_dataset, header=None))}')

Novos casos Tem jogo adicionados: 0
Total final de linhas: 5478


In [16]:
import pandas as pd

# Setor dedicado: balanceador automatico de todas as classes do dataset.
arquivo_entrada = 'dataset_tratado_balanceado.csv'
arquivo_saida = 'dataset_tratado_balanceado_auto.csv'
random_state = 42

# Se quiser manter menos linhas, troque para 'undersample'.
# 'oversample' replica classes menores ate o tamanho da classe majoritaria.
# 'undersample' reduz classes maiores ate o tamanho da classe minoritaria.
modo_balanceamento = 'oversample'

df = pd.read_csv(arquivo_entrada, header=None)
coluna_classe = df.columns[-1]

distribuicao_antes = df[coluna_classe].value_counts()

if modo_balanceamento == 'undersample':
    alvo = distribuicao_antes.min()
    replace = False
else:
    alvo = distribuicao_antes.max()
    replace = True

partes_balanceadas = []
for classe, grupo in df.groupby(coluna_classe):
    precisa_repor = replace and len(grupo) < alvo
    grupo_balanceado = grupo.sample(n=alvo, replace=precisa_repor, random_state=random_state)
    partes_balanceadas.append(grupo_balanceado)

df_balanceado = pd.concat(partes_balanceadas, ignore_index=True)
df_balanceado = df_balanceado.sample(frac=1, random_state=random_state).reset_index(drop=True)

df_balanceado.to_csv(arquivo_saida, index=False, header=False)

print('Balanceamento automatico concluido!')
print(f'Arquivo de entrada: {arquivo_entrada}')
print(f'Arquivo de saida: {arquivo_saida}')
print('\nDistribuicao antes:')
print(distribuicao_antes)
print('\nDistribuicao depois:')
print(df_balanceado[coluna_classe].value_counts())
print(f'\nTotal final de linhas: {len(df_balanceado)}')

Balanceamento automatico concluido!
Arquivo de entrada: dataset_tratado_balanceado.csv
Arquivo de saida: dataset_tratado_balanceado_auto.csv

Distribuicao antes:
9
Possibilidade_fim    2439
Tem jogo             2081
Fim_X_Vence           626
Empate                500
Fim_O_Vence           316
Fim_Empate             16
Name: count, dtype: int64

Distribuicao depois:
9
Empate               2439
Fim_O_Vence          2439
Possibilidade_fim    2439
Fim_X_Vence          2439
Tem jogo             2439
Fim_Empate           2439
Name: count, dtype: int64

Total final de linhas: 14634


In [19]:
import pandas as pd
import itertools

# Setor dedicado: adicionar X casos de empate no dataset.
arquivo_dataset = 'dataset_tratado_balanceado.csv'
X = 500  # Ajuste aqui quantos empates deseja adicionar.
random_state = 42

linhas_vitoria = [
    [0, 1, 2], [3, 4, 5], [6, 7, 8],
    [0, 3, 6], [1, 4, 7], [2, 5, 8],
    [0, 4, 8], [2, 4, 6]
]


def eh_empate(board):
    # Empate final valido: 5 X, 4 O, tabuleiro cheio e sem vencedor.
    if board.count(1) != 5 or board.count(-1) != 4 or board.count(0) != 0:
        return False

    for l in linhas_vitoria:
        soma = board[l[0]] + board[l[1]] + board[l[2]]
        if soma == 3 or soma == -3:
            return False

    return True


# Gera todos os tabuleiros de empate finais possiveis (sem simulacao de partida).
base_jogadas = [1, 1, 1, 1, 1, -1, -1, -1, -1]
combinacoes = set(itertools.permutations(base_jogadas))
casos_empate = [list(b) for b in combinacoes if eh_empate(list(b))]

df_empates_base = pd.DataFrame(casos_empate)

if df_empates_base.empty:
    print('Nenhum caso de empate foi gerado.')
else:
    df_dataset = pd.read_csv(arquivo_dataset, header=None)
    df_empates_novos = df_empates_base.sample(n=X, replace=True, random_state=random_state).copy()
    df_empates_novos[9] = 'Empate'

    df_resultado = pd.concat([df_dataset, df_empates_novos], ignore_index=True)
    df_resultado.to_csv(arquivo_dataset, index=False, header=False)

    print(f'Casos de empate adicionados: {X}')
    print(f'Total final de linhas: {len(df_resultado)}')

Casos de empate adicionados: 500
Total final de linhas: 5978


In [34]:
import pandas as pd

# Setor dedicado: remover N linhas aleatorias de uma classe especifica (não remove por classe inteira).
# Configure:
arquivo_dataset = 'dataset_tratado_balanceado.csv'  # arquivo alvo
classe_alvo = 'Fim_Empate'  # rótulo da classe a remover (string exata, case-insensitive)
N = 16  # quantas linhas remover
random_state = 42

# Leitura
df = pd.read_csv(arquivo_dataset, header=None)
col_classe = df.columns[-1]

# Encontrar indices da classe alvo (comparação case-insensitive)
mask = df[col_classe].astype(str).str.strip().str.lower() == str(classe_alvo).strip().lower()
indices = df[mask].index
qtd_disponivel = len(indices)

if qtd_disponivel == 0:
    print(f"Nenhuma linha encontrada para a classe '{classe_alvo}'. Nada foi removido.")
else:
    n_remover = min(N, qtd_disponivel)
    indices_remover = df.loc[indices].sample(n=n_remover, random_state=random_state).index

    df_filtrado = df.drop(index=indices_remover).reset_index(drop=True)
    df_filtrado.to_csv(arquivo_dataset, index=False, header=False)

    print(f"Classe alvo: {classe_alvo}")
    print(f"Disponivel: {qtd_disponivel}")
    print(f"Removidas: {n_remover}")
    print(f"Total final de linhas: {len(df_filtrado)}")
    
    # Opcional: mostrar quais indices foram removidos (antes do reset)
    print('Indices removidos (originais):', list(indices_remover))


Classe alvo: Fim_Empate
Disponivel: 16
Removidas: 16
Total final de linhas: 984
Indices removidos (originais): [37, 43, 100, 342, 331, 291, 146, 287, 44, 400, 96, 113, 289, 327, 95, 107]
